# FINN - Notebook Miguel - Red cuantizada 16x12 RGB → 2 salidas

Este notebook está adaptado para tu red propia guardada como:

`state_dict_quantized_last.pth`

La red trabaja con imágenes RGB de `16 x 12`, por tanto la entrada es:

`(1, 3, 16, 12)`

Como `3 x 16 x 12 = 576`, la primera capa lineal tiene entrada de 576 valores.

La salida de la red es de tamaño 2, normalmente interpretable como:

- `out0` → velocidad lineal `v`
- `out1` → velocidad angular `w`

In [ ]:
# 1. Imports y preparación básica
import os
import sys
import gc
import warnings

warnings.filterwarnings("default")
gc.collect()

print("Python path actual:", sys.path[0])
print("Carpeta actual:", os.getcwd())

In [ ]:
# 2. Configurar FINN_BUILD_DIR
# FINN necesita esta variable para crear carpetas temporales de generación.

FINN_BUILD_DIR = os.path.abspath("FINN_build")
os.environ["FINN_BUILD_DIR"] = FINN_BUILD_DIR
os.makedirs(FINN_BUILD_DIR, exist_ok=True)

print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])

In [ ]:
# 3. Imports de PyTorch, Brevitas, QONNX y FINN
import torch
import torch.nn as nn

import brevitas.nn as qnn
from brevitas.quant import Int8Bias
import brevitas.onnx as bo

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

from finn.util.visualization import showInNetron
from finn.util.basic import make_build_dir

print("Torch cargado. CUDA disponible:", torch.cuda.is_available())

In [ ]:
# 4. Parámetros de la red propia

WEIGHTS_FILE = "state_dict_quantized_last.pth"
MODEL_ONNX = "model.onnx"
MODEL_TIDY_ONNX = "model_tidy.onnx"

# MUY IMPORTANTE:
# Tu red NO usa 64x64 en gris.
# Tu red usa 16x12 RGB.
INPUT_SHAPE = (1, 3, 16, 12)

INPUT_FEATURES = 3 * 16 * 12
NUM_OUTPUTS = 2

print("Archivo de pesos:", WEIGHTS_FILE)
print("INPUT_SHAPE:", INPUT_SHAPE)
print("INPUT_FEATURES:", INPUT_FEATURES)
print("NUM_OUTPUTS:", NUM_OUTPUTS)

assert INPUT_FEATURES == 576
assert os.path.exists(WEIGHTS_FILE), f"No encuentro {WEIGHTS_FILE}. Súbelo a la misma carpeta del notebook."

In [ ]:
# 5. Inspección del state_dict
# Esta celda sirve para confirmar que los pesos coinciden con la red esperada.

state_dict_check = torch.load(WEIGHTS_FILE, map_location="cpu")

if isinstance(state_dict_check, dict) and "state_dict" in state_dict_check:
    state_dict_check = state_dict_check["state_dict"]

for k, v in state_dict_check.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v))

In [ ]:
# 6. Definición de la arquitectura real de tu red
# Estructura detectada desde el state_dict:
# 576 -> 256 -> 128 -> 16 -> 2

class RedMiguel(nn.Module):
    def __init__(self, in_width=8, weight_width=4, act_width=4, out_width=8):
        super(RedMiguel, self).__init__()

        self.quant_input = qnn.QuantIdentity(
            return_quant_tensor=True,
            bit_width=in_width
        )

        self.fc1 = qnn.QuantLinear(
            in_features=576,
            out_features=256,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu1 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.fc2 = qnn.QuantLinear(
            in_features=256,
            out_features=128,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu2 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            in_features=128,
            out_features=16,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu3 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            in_features=16,
            out_features=2,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.quant_output = qnn.QuantIdentity(
            return_quant_tensor=True,
            bit_width=out_width
        )

    def forward(self, x):
        x = self.quant_input(x)

        # Entrada esperada: (batch, 3, 16, 12)
        # Se aplana a 576 valores: RRR... GGG... BBB...
        x = x.reshape(x.size(0), -1)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fx(x)
        x = self.relu3(x)

        x = self.predictor(x)
        x = self.quant_output(x)

        return x

print("Arquitectura RedMiguel definida correctamente.")

In [ ]:
# 7. Crear modelo y cargar los pesos propios

model = RedMiguel(in_width=8, weight_width=4, act_width=4, out_width=8)

state_dict = torch.load(WEIGHTS_FILE, map_location="cpu")

if isinstance(state_dict, dict) and "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

model.load_state_dict(state_dict)
model.eval()

print("Pesos cargados correctamente desde:", WEIGHTS_FILE)

In [ ]:
# 8. Probar una inferencia dummy en PyTorch
# Si esta celda falla con 4096 vs 576, significa que INPUT_SHAPE sigue mal.

with torch.no_grad():
    dummy_input = torch.randn(*INPUT_SHAPE)
    dummy_output = model(dummy_input)

print("Forma entrada:", tuple(dummy_input.shape))
print("Forma salida:", tuple(dummy_output.shape))
print("Salida dummy:", dummy_output)

In [ ]:
# 9. Exportar la red a ONNX compatible con FINN

bo.export_finn_onnx(model, INPUT_SHAPE, MODEL_ONNX)
print("Modelo exportado a:", MODEL_ONNX)

In [ ]:
# 10. Comprobar entrada y salida del ONNX

model_check = ModelWrapper(MODEL_ONNX)

print("Entradas:")
for inp in model_check.graph.input:
    print(inp.name, model_check.get_tensor_shape(inp.name))

print("
Salidas:")
for out in model_check.graph.output:
    print(out.name, model_check.get_tensor_shape(out.name))

In [ ]:
# 11. Limpieza básica del ONNX: nombres, shapes y constantes

model_qonnx = ModelWrapper(MODEL_ONNX)
model_qonnx = model_qonnx.transform(InferShapes())
model_qonnx = model_qonnx.transform(FoldConstants())
model_qonnx = model_qonnx.transform(GiveUniqueNodeNames())
model_qonnx = model_qonnx.transform(GiveReadableTensorNames())
model_qonnx = model_qonnx.transform(RemoveStaticGraphInputs())
model_qonnx.save(MODEL_TIDY_ONNX)

print("Modelo ONNX limpio guardado en:", MODEL_TIDY_ONNX)

In [ ]:
# 12. Mostrar estructura del modelo limpio

model_check = ModelWrapper(MODEL_TIDY_ONNX)

print("Entradas:")
for inp in model_check.graph.input:
    print(inp.name, model_check.get_tensor_shape(inp.name))

print("
Salidas:")
for out in model_check.graph.output:
    print(out.name, model_check.get_tensor_shape(out.name))

print("
Nodos:")
for node in model_check.graph.node:
    print(node.op_type, node.name)

In [ ]:
# 13. Visualización opcional con Netron
# Descomenta si quieres verlo en Netron.

# showInNetron(MODEL_TIDY_ONNX)

## A partir de aquí empieza el flujo de FINN build_dataflow

Esta parte intenta generar los resultados en `output_final_miguel`.

Si tu versión del notebook original del profesor usaba más pasos concretos, puedes copiar esas celdas debajo de esta sección usando como entrada `model_tidy.onnx` o `model.onnx`, según corresponda.

In [ ]:
# 14. Configuración build_dataflow de FINN

import shutil
from finn.builder import build_dataflow as build
from finn.builder.build_dataflow_config import DataflowOutputType

final_output_dir = "output_final_miguel"

if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("Previous run results deleted:", final_output_dir)

# Selección del fichero folding.
# Para tu red totalmente conectada puede que el folding_config_3cnv.json del ejemplo no encaje.
# Por eso primero intentamos sin folding_config_file fijo.
# Si necesitas usar uno concreto, cambia folding_config_file=None por "folding_config_3cnv.json".

cfg = build.DataflowBuildConfig(
    output_dir=final_output_dir,
    folding_config_file=None,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    target_fps=1000000,
    synth_clk_period_ns=10.0,
    board=None,
    shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ if False else None
)

print("Configuración creada. Salida:", final_output_dir)

In [ ]:
# 15. Ejecutar build_dataflow
# Esta celda puede tardar bastante.
# Si falla por configuración de folding, primero pasa el error y se ajusta el folding_config.

build.build_dataflow_cfg(MODEL_TIDY_ONNX, cfg)
print("Build finalizado. Revisa la carpeta:", final_output_dir)

In [ ]:
# 16. Buscar archivos importantes generados

import os

print("Buscando archivos Verilog / Vivado / all_verilog_srcs...")
found = False
for root, dirs, files in os.walk(final_output_dir):
    for f in files:
        if (
            "all_verilog_srcs" in f
            or "finn_design_wrapper" in f
            or f.endswith(".v")
            or f.endswith(".sv")
            or f.endswith(".dcp")
        ):
            print(os.path.join(root, f))
            found = True

if not found:
    print("No se han encontrado aún Verilog/DCP. Puede que el build solo haya generado modelos intermedios/reportes o que falte activar más outputs.")

In [ ]:
# 17. Mostrar árbol resumido de la carpeta de salida

for root, dirs, files in os.walk(final_output_dir):
    level = root.replace(final_output_dir, "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")
    for f in files[:10]:
        print(indent + "  " + f)

## Resumen para el TFG

Con este notebook se hace:

`state_dict_quantized_last.pth` → carga de pesos en PyTorch → exportación a `model.onnx` → limpieza ONNX → preparación para FINN → generación en `output_final_miguel`.

La red usada en este notebook es tu red real:

`576 → 256 → 128 → 16 → 2`

con entrada:

`1 x 3 x 16 x 12`

Esto coincide con el formato de simulación usado en C++:

`RRR... GGG... BBB...`